In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import time
import random
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
df_train=pd.read_csv("../data/retail_sales_kaggle/train.csv")
df_test=pd.read_csv("../data/retail_sales_kaggle/test.csv")

In [3]:
display(df_train)
print("\n\n",df_train["item"].unique(),"\n\n")
print("\n\n",df_train.columns,"\n\n")
print("\n\n",df_train.info(),"\n\n")

,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10
...,...,...,...,...
912995,2017-12-27,10,50,63
912996,2017-12-28,10,50,59
912997,2017-12-29,10,50,74
912998,2017-12-30,10,50,62




 [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50] 




 Index(['date', 'store', 'item', 'sales'], dtype='object') 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB


 None 




In [4]:
df_train.drop("store",axis=1,inplace=True)
df_test.drop("store",axis=1,inplace=True)
df_train["date"] = pd.to_datetime(df_train['date'])
df_test["date"] = pd.to_datetime(df_test['date'])

top_items_series = df_train.groupby('item')['sales'].sum().sort_values(ascending=False)
top_30_item_ids = top_items_series.head(30).index.tolist()
df_train = df_train[df_train['item'].isin(top_30_item_ids)].copy()
df_test = df_test[df_test['item'].isin(top_30_item_ids)].copy()
print(f"Seçilen Popüler Ürünler: {top_30_item_ids}")
print(f"Yeni Eğitim Seti Boyutu: {df_train.shape}")

def create_features(df):
    df['month'] = df['date'].dt.month
    df['day_of_month'] = df['date'].dt.day
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['year'] = df['date'].dt.year
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    return df

df_train = create_features(df_train)
df_test = create_features(df_test)

Seçilen Popüler Ürünler: [15, 28, 13, 18, 25, 45, 38, 22, 36, 8, 10, 11, 12, 29, 33, 24, 50, 35, 14, 31, 46, 2, 7, 6, 9, 48, 43, 26, 20, 32]
Yeni Eğitim Seti Boyutu: (547800, 3)


In [5]:
my_products=["Kars Kaşarı","Erzincan Tulumu","İzmir Tulumu","Çeçil Peyniri","Süzme Yoğurt","Manda Yoğurdu","Meyveli Yoğurt",
             "Keçi Yoğurdu","Tam Yağlı Süt","Yarım Yağlı Süt","Laktozsuz Süt","Yayık Tereyağı","Vakfıkebir Tereyağı","Köy Tereyağı",
             "Tuzlu Tereyağı","Maraş Dondurması","Vanilyalı Dondurma","Kakaolu Dondurma","Sade Yağ","Süt Yağı","Naneli Ayran","Sade Ayran",
             "Fesleğenli Ayran","Lor Peyniri","Köy Peyniri","Süzme Peynir","Çökelek","Kefir","Pastörize Ayran","Kımız"]

random.seed(42)
shuffled_products = my_products.copy()
random.shuffle(shuffled_products)

mapping_dict = dict(zip(top_30_item_ids, shuffled_products))
df_train['Product Name'] = df_train['item'].map(mapping_dict)
print("Eşleştirme Örneği:", list(mapping_dict.items())[:5])

Eşleştirme Örneği: [(15, 'Süt Yağı'), (28, 'Tuzlu Tereyağı'), (13, 'Laktozsuz Süt'), (18, 'Çökelek'), (25, 'Fesleğenli Ayran')]


In [6]:
df_train = df_train.sort_values(by=['item', 'date'])

def add_lags(df):
    df['sales_lag_1'] = df.groupby('item')['sales'].shift(1)
    df['sales_lag_7'] = df.groupby('item')['sales'].shift(7)
    df['sales_lag_30'] = df.groupby('item')['sales'].shift(30)
    df['sales_roll_mean_7'] = df.groupby('item')['sales'].transform(lambda x: x.shift(1).rolling(window=7).mean())
    return df
    
df_train = add_lags(df_train)
df_train.dropna(inplace=True)

In [7]:
train_final = df_train[df_train['date'] < '2017-07-01']
val_final = df_train[df_train['date'] >= '2017-07-01']

X_train = train_final.drop(['sales', 'date'], axis=1)
y_train = train_final['sales']

X_val = val_final.drop(['sales', 'date'], axis=1)
y_val = val_final['sales']

In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_train = ohe.fit_transform(X_train[['Product Name']])
ohe_val = ohe.transform(X_val[['Product Name']])

ohe_cols = ohe.get_feature_names_out(['Product Name'])
X_train_ohe = pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
X_val_ohe = pd.DataFrame(ohe_val, columns=ohe_cols, index=X_val.index)

X_train_final = pd.concat([X_train.drop(['item', 'Product Name'], axis=1), X_train_ohe], axis=1)
X_val_final = pd.concat([X_val.drop(['item', 'Product Name'], axis=1), X_val_ohe], axis=1)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_val_scaled = scaler.transform(X_val_final)

print(f"Final Sütun Sayısı: {X_train_final.shape[1]}")

Final Sütun Sayısı: 43


In [9]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    # "SVR (RBF)": SVR(kernel="rbf", C=100, epsilon=0.1), Yavaş
    "Random Forest": RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, n_jobs=-1, random_state=42), 
    #"Gradient Boosting": GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, random_state=42),  0.847 r2 score ile 330 saniyede eğitti
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42, tree_method='hist'),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, verbose=0, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    
    end_time = time.time()
    
    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    duration = end_time - start_time
    
    results.append({
        "Model": name,
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse,
        "Time (sec)": f"{duration:.4f}"
    })
    print(f"{name} tamamlandı. (R2: {r2:.4f})")

df_results = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
display(df_results)

Linear Regression tamamlandı. (R2: 0.7494)
Ridge Regression tamamlandı. (R2: 0.7496)
Lasso Regression tamamlandı. (R2: 0.7493)
Random Forest tamamlandı. (R2: 0.8499)
Extra Trees tamamlandı. (R2: 0.8478)
XGBoost tamamlandı. (R2: 0.8490)
CatBoost tamamlandı. (R2: 0.8402)
LightGBM tamamlandı. (R2: 0.8519)


,Model,R2 Score,MAE,RMSE,Time (sec)
7,LightGBM,0.851862,8.574299,11.001631,5.1903
3,Random Forest,0.849920,8.551483,11.073514,59.5676
5,XGBoost,0.848990,8.656375,11.107757,4.2487
4,Extra Trees,0.847832,8.614481,11.150286,71.1372
6,CatBoost,0.840215,8.915338,11.425945,16.3459
1,Ridge Regression,0.749597,11.207926,14.303546,0.1438
0,Linear Regression,0.749432,11.215292,14.308272,0.5682
2,Lasso Regression,0.749314,11.168232,14.311625,0.3891


In [10]:
final_model=LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=299, verbose=-1)

final_model.fit(X_train_scaled, y_train)
y_pred = final_model.predict(X_val_scaled)

r2 = r2_score(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("-" * 30)
print("FINAL MODEL PERFORMANCE")
print("R2 Score:",r2)
print("MAE:",mae)
print("RMSE:",rmse)
print("-" * 30)



------------------------------
FINAL MODEL PERFORMANCE
R2 Score: 0.8515397579899033
MAE: 8.583068932337671
RMSE: 11.01359888236916
------------------------------


In [11]:
import pickle
import os

model_pack = {
    "model": final_model,
    "ohe": ohe,                
    "scaler": scaler,          
    "features": list(X_train_final.columns), 
    "mapping": mapping_dict, 
    "metrics": {
        "r2": 0.8519,
        "mae": 8.57,
        "description": "Kaggle Retail Sales Data - Top 30 Items - Time Series Model"
    }
}

file_name = "team299_lgbm_final.pkl"
export_folder = os.path.join('..', 'exports')

if not os.path.exists(export_folder):
    os.makedirs(export_folder)

full_path = os.path.join(export_folder, file_name)

try:
    with open(full_path, "wb") as f:
        pickle.dump(model_pack, f)
        
    print(f" Konum: {os.path.abspath(full_path)}")
    print(f" Paket içeriği: Model, OHE, Scaler, Feature List ve Mapping")

except Exception as e:
    print(f"Export sırasında hata: {e}")

 Konum: C:\Users\Bariscan\hackathon2026\YZTA-Hackathon-Team299\ml_module\exports\team299_lightgbm_final.pkl
 Paket içeriği: Model, OHE, Scaler, Feature List ve Mapping
